In [ ]:
source("helpers/init.r")

In [ ]:
cohorts    <- fread(paste0(READY_DATA, "cohorts.txt"))
plt_cohort <- cohorts %>% filter(!ref) %>% pull(cohort) %>% unique()
ref_cohort <- cohorts %>% filter(ref)  %>% pull(cohort) %>% unique()

# 0 - Load data

In [ ]:
alloc_col_map <- c(
  allocationAging            = "Ageing",
  allocationApobec           = "APOBEC",
  allocationChemicalExposure = "Chemical exposure",
  allocationDnaDamageRepair  = "DNA damage repair",
  allocationTobacco          = "Tobacco",
  allocationUvLight          = "UV light"
)

alloc_cols <- names(alloc_col_map)

raw    <- fread(paste0(BASE_DATA, "markers.txt")) %>% select(sampleId, all_of(alloc_cols))

In [ ]:
validate_columns(raw, c("sampleId", "allocationAging", "allocationApobec",
                     "allocationDnaDamageRepair", "allocationTobacco",
                     "allocationUvLight", "allocationChemicalExposure"), "markers (signatures)")

# 1 - Compute cohort-level contributions

In [ ]:
sigs_ready <-
  cohorts %>%
  left_join(raw, by = "sampleId") %>%
  pivot_longer(all_of(alloc_cols), names_to = "feature", values_to = "allocation") %>%
  mutate(gp = alloc_col_map[feature]) %>%
  group_by(cohort, gp) %>%
  summarise(total = sum(allocation, na.rm = TRUE), .groups = "drop") %>%
  group_by(cohort) %>%
  mutate(pct  = round(100 * total / sum(total), 1),
         spot = row_number(pct)) %>%
  ungroup()

# 2 - Circular plot

In [ ]:
make_sig_plot <- function(cohort_name) {
  go <- sigs_ready %>%
    mutate(spot = ifelse(cohort == ref_cohort, spot + .3, spot)) %>%
    rowwise() %>%
    mutate(col = ifelse(cohort == cohort_name,
                        sig_colors[[gp]], fills[["ref"]])) %>%
    ungroup() %>%
    mutate(cohort = factor(cohort, levels = c(ref_cohort, cohort_name)))

  max_x <- max(go$pct, na.rm = TRUE) * 1.35

  ggplot(go, aes(x = 0, xend = pct, y = spot, yend = spot, color = col)) +
    geom_segment(color = "#bbbbbb", linewidth = 3.5, lineend = "round", alpha = .9) +
    geom_segment(linewidth = 2.5, lineend = "round") +
    scale_color_identity() +
    scale_y_continuous(limits = c(0, max(go$spot) + 0.5)) +
    scale_x_continuous(limits = c(0, max_x)) +
    geom_text(
      data  = subset(go, cohort == cohort_name),
      size  = 3.5, hjust = "right", color = "black",
      aes(x = 0, y = spot, label = paste0(gp, ": ", as.character(round(pct, 0)), "%    "))
    ) +
    coord_polar() +
    theme_void() +
    theme(plot.margin = margin(-1, -1, -1, -1, "cm"))
}

sig_plt <- make_sig_plot(plt_cohort)

# 3 - Save

In [ ]:
saveRDS(list(signatures = sig_plt), paste0(READY_FIG, "signatures.rds"))